# 02 · Exploring the raw files with DuckDB

Goal: design the staging layer. Each raw file is read with SQL to check how DuckDB
infers types, how missing data shows up, and how to reshape everything into two
long-format tables: `stg_prices` and `stg_fred`.

In [64]:
import duckdb 
from pathlib import Path

In [65]:
con = duckdb.connect()
SPY_PATH = 'C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_SPY.csv'

## Price CSVs: type inference

DuckDB infers the types correctly (`Date` → DATE, prices → DOUBLE, `Volume` → BIGINT).
Even so, `src/load.py` casts every column explicitly so the schema doesn't depend
on auto-detection. `Adj Close` contains a space, so it's renamed to `adj_close`.

In [66]:
duckdb.sql("SELECT * FROM read_csv('C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_SPY.csv') LIMIT 5")

DateDATE,Adj CloseDOUBLE,CloseDOUBLE,HighDOUBLE,LowDOUBLE,OpenDOUBLE,VolumeBIGINT
2005-01-03,80.9735336303711,120.30000305175781,121.76000213623047,119.9000015258789,121.55999755859375,55748000
2005-01-04,79.98407745361328,118.83000183105469,120.54000091552734,118.44000244140625,120.45999908447266,69167600
2005-01-05,79.43213653564453,118.01000213623047,119.25,118.0,118.73999786376953,65667300
2005-01-06,79.83601379394531,118.61000061035156,119.1500015258789,118.26000213623047,118.44000244140625,47814700
2005-01-07,79.72159576416016,118.44000244140625,119.2300033569336,118.12999725341797,118.97000122070312,55847700


In [67]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_csv('{SPY_PATH}')") 

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
Date,DATE,YES,NULL,NULL,NULL
Adj Close,DOUBLE,YES,NULL,NULL,NULL
Close,DOUBLE,YES,NULL,NULL,NULL
High,DOUBLE,YES,NULL,NULL,NULL
Low,DOUBLE,YES,NULL,NULL,NULL
Open,DOUBLE,YES,NULL,NULL,NULL
Volume,BIGINT,YES,NULL,NULL,NULL


## DBC: missing values depend on data structure

When the six tickers were downloaded together (wide format), DBC had 23 nulls in
Jan–Feb 2006. Downloaded alone, it has none: it simply starts on 2006-02-06.
The missing data didn't disappear; it changed shape, from **null cells** (wide)
to **missing rows** (long). Validation must therefore check date ranges, not only nulls.

In [68]:
DBC_PATH = 'C:/Users/ignac/Proyectos/inflation-hedge-analysis/data/raw/prices_DBC.csv'

duckdb.sql(f"SELECT MAX(DATE) FROM read_csv('{DBC_PATH}')")

max(DATE)DATE
2026-10-30


In [ ]:
duckdb.sql(f"""
    SELECT
        MIN(Date)                       AS first_date,
        MAX(Date)                       AS last_date,
        COUNT(*)                        AS n_rows,
        COUNT(*) - COUNT("Adj Close")   AS null_adj_close
    FROM read_csv('{DBC_PATH}')
""").df()

,first_date,last_date,n_rows,null_adj_close
0,2006-02-06,2026-10-30,5217,0


In [69]:
duckdb.sql(f"SELECT MIN(DATE) FROM read_csv('{DBC_PATH}')")

min(DATE)DATE
2006-02-06


In [70]:
duckdb.sql(f"SELECT COUNT(*) FROM read_csv('{DBC_PATH}')")

count_star()BIGINT
5217


In [71]:
duckdb.sql(f"""
    SELECT COUNT(*) - COUNT("Adj Close") AS null_adj_close
    FROM read_csv('{DBC_PATH}')
""")

null_adj_closeBIGINT
0


In [72]:
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"

cpi_path = RAW_DIR / "fred_CPIAUCSL.json"

In [73]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_json('{cpi_path.as_posix()}')").df()

,column_name,column_type,null,key,default,extra
0,realtime_start,DATE,YES,None,None,None
1,realtime_end,DATE,YES,None,None,None
2,observation_start,DATE,YES,None,None,None
3,observation_end,DATE,YES,None,None,None
4,units,VARCHAR,YES,None,None,None
5,output_type,BIGINT,YES,None,None,None
6,file_type,VARCHAR,YES,None,None,None
7,order_by,VARCHAR,YES,None,None,None
8,sort_order,VARCHAR,YES,None,None,None
9,count,BIGINT,YES,None,None,None


## FRED JSON: unnesting observations

Each file is a single row whose `observations` column is a list of structs
(`STRUCT(...)[]`). `UNNEST` turns that list into one row per month.
The `count` field returned by FRED (261 per series at download time) can be
used to confirm nothing was lost.

In [74]:
duckdb.sql(f" SELECT UNNEST(observations, recursive := true) FROM read_json('{cpi_path.as_posix()}')")

realtime_startDATE,realtime_endDATE,dateDATE,valueVARCHAR
2026-10-22,2026-10-22,2005-01-01,191.600
2026-10-22,2026-10-22,2005-02-01,192.400
2026-10-22,2026-10-22,2005-03-01,193.100
2026-10-22,2026-10-22,2005-04-01,193.700
2026-10-22,2026-10-22,2005-05-01,193.600
2026-10-22,2026-10-22,2005-06-01,193.700
2026-10-22,2026-10-22,2005-07-01,194.900
2026-10-22,2026-10-22,2005-08-01,196.100
2026-10-22,2026-10-22,2005-09-01,198.800
2026-10-22,2026-10-22,2005-10-01,199.100


In [75]:
duckdb.sql(f"DESCRIBE SELECT UNNEST(observations, recursive := true) FROM read_json('{cpi_path.as_posix()}')")

column_nameVARCHAR,column_typeVARCHAR,nullVARCHAR,keyVARCHAR,defaultVARCHAR,extraVARCHAR
realtime_start,DATE,YES,NULL,NULL,NULL
realtime_end,DATE,YES,NULL,NULL,NULL
date,DATE,YES,NULL,NULL,NULL
value,VARCHAR,YES,NULL,NULL,NULL


In [76]:
duckdb.sql(f"""
    SELECT
        obs.date  AS date,
        TRY_CAST(obs.value AS DOUBLE) AS value
    FROM (
        SELECT UNNEST(observations) AS obs
        FROM read_json('{cpi_path.as_posix()}')
    ) AS t
""")

dateDATE,valueDOUBLE
2005-01-01,191.6
2005-02-01,192.4
2005-03-01,193.1
2005-04-01,193.7
2005-05-01,193.6
2005-06-01,193.7
2005-07-01,194.9
2005-08-01,196.1
2005-09-01,198.8
2005-10-01,199.1


## Handling FRED's missing-value marker

FRED stores values as strings and marks missing data with `"."`
(e.g. CPI for October 2025, not published due to the US government shutdown).

- `TRY_CAST` would turn *any* unparseable value into NULL silently.
- `CAST(NULLIF(value, '.') AS DOUBLE)` converts only the known marker to NULL;
  any other unexpected value makes the query fail (fail fast).

Note: an equivalent `CASE WHEN value = '.' THEN NULL ELSE CAST(...) END` raised a
conversion error in DuckDB when accessing a struct field, so `NULLIF` is used instead.

In [77]:
# Casting value to be DOBLE in CPI 

duckdb.sql(f"""
    SELECT
        obs.date, CAST(NULLIF(obs.value, '.') AS DOUBLE) AS value
    FROM (
        SELECT UNNEST(observations) AS obs
        FROM read_json('{cpi_path.as_posix()}')
    ) AS t
""")

dateDATE,valueDOUBLE
2005-01-01,191.6
2005-02-01,192.4
2005-03-01,193.1
2005-04-01,193.7
2005-05-01,193.6
2005-06-01,193.7
2005-07-01,194.9
2005-08-01,196.1
2005-09-01,198.8
2005-10-01,199.1


## Long format in one query

`read_json`/`read_csv` accept wildcards (`fred_*.json`, `prices_*.csv`).
With `filename = true`, each row keeps its source file, and `regexp_extract`
recovers the series ID or ticker from the file name. This produces the
long-format staging tables used by `src/load.py`.

In [78]:
#Extracting series_id from FILENAME in FRED FILES

duckdb.sql(rf"""
    SELECT
        regexp_extract(filename, 'fred_(.+)\.json', 1) AS series_id,
        obs.date                                       AS date,
        CAST(NULLIF(obs.value, '.') AS DOUBLE)         AS value
    FROM (
        SELECT filename, UNNEST(observations) AS obs
        FROM read_json('{RAW_DIR.as_posix()}/fred_*.json', filename = true)
    ) AS t
""")

series_idVARCHAR,dateDATE,valueDOUBLE
CPIAUCSL,2005-01-01,191.6
CPIAUCSL,2005-02-01,192.4
CPIAUCSL,2005-03-01,193.1
CPIAUCSL,2005-04-01,193.7
CPIAUCSL,2005-05-01,193.6
CPIAUCSL,2005-06-01,193.7
CPIAUCSL,2005-07-01,194.9
CPIAUCSL,2005-08-01,196.1
CPIAUCSL,2005-09-01,198.8
CPIAUCSL,2005-10-01,199.1


In [79]:
# Counting how many rows each series has

duckdb.sql(rf"""
    SELECT
        regexp_extract(filename, 'fred_(.+)\.json', 1) AS series_id,
        COUNT(*)
    FROM (
        SELECT filename, UNNEST(observations) AS obs
        FROM read_json('{RAW_DIR.as_posix()}/fred_*.json', filename = true)
    ) AS t
    GROUP BY series_id
""")

series_idVARCHAR,count_star()BIGINT
CPIAUCSL,261
TB3MS,261


In [80]:
#Extracting ticker_id from all the csvs

duckdb.sql(rf"""
    SELECT
        Date as date,
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker,
        "Adj Close" AS adj_close,
        Close       AS close,
        High        AS high,
        Low         AS low,
        Open        AS open,
        Volume      AS volume
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

dateDATE,tickerVARCHAR,adj_closeDOUBLE,closeDOUBLE,highDOUBLE,lowDOUBLE,openDOUBLE,volumeBIGINT
2006-02-06,DBC,19.185100555419922,24.200000762939453,24.65999984741211,24.100000381469727,24.549999237060547,771500
2006-02-07,DBC,18.630155563354492,23.5,24.200000762939453,23.450000762939453,24.049999237060547,674400
2006-02-08,DBC,18.550880432128906,23.399999618530273,23.670000076293945,23.329999923706055,23.489999771118164,335400
2006-02-09,DBC,18.725292205810547,23.6200008392334,23.81999969482422,23.510000228881836,23.559999465942383,705900
2006-02-10,DBC,18.384395599365234,23.190000534057617,23.690000534057617,23.100000381469727,23.649999618530273,276700
2006-02-13,DBC,18.098997116088867,22.829999923706055,23.200000762939453,22.770000457763672,23.190000534057617,237600
2006-02-14,DBC,17.9483699798584,22.639999389648438,22.829999923706055,22.610000610351562,22.799999237060547,256400
2006-02-15,DBC,17.686758041381836,22.309999465942383,22.729999542236328,22.239999771118164,22.700000762939453,404800
2006-02-16,DBC,17.900806427001953,22.579999923706055,22.6200008392334,22.25,22.329999923706055,463600
2006-02-17,DBC,18.154491424560547,22.899999618530273,22.920000076293945,22.68000030517578,22.719999313354492,467700


In [81]:
#Total Rows

duckdb.sql(rf"""
    SELECT
        COUNT(regexp_extract(filename, 'prices_(.+)\.csv', 1)) AS TICKER
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

TICKERBIGINT
32677


In [82]:
# How many rows per ticker

duckdb.sql(rf"""
    SELECT
        COUNT(*),
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
    GROUP BY ticker
""")

count_star()BIGINT,tickerVARCHAR
5492,SPY
5492,GLD
5217,DBC
5492,VNQ
5492,TIP
5492,TLT


In [83]:
#View that will be utiliced for checking nulls

duckdb.sql(rf"""
    CREATE OR REPLACE VIEW stg_prices_preview AS
    SELECT
        Date                                            AS date,
        regexp_extract(filename, 'prices_(.+)\.csv', 1) AS ticker,
        "Adj Close"                                     AS adj_close,
        Close                                           AS close,
        High                                            AS high,
        Low                                             AS low,
        Open                                            AS open,
        Volume                                          AS volume
    FROM read_csv('{RAW_DIR.as_posix()}/prices_*.csv', filename = true)
""")

CountBIGINT


In [84]:
duckdb.sql("SELECT COUNT(*) AS total_rows FROM stg_prices_preview")

total_rowsBIGINT
32677


In [85]:
duckdb.sql("""
    SELECT
        ticker,
        COUNT(*),
        MIN(DATE) as first_date, 
        MAX(DATE) as last_date,
    FROM stg_prices_preview
    GROUP BY ticker    
    ORDER BY ticker
""")

tickerVARCHAR,count_star()BIGINT,first_dateDATE,last_dateDATE
DBC,5217,2006-02-06,2026-10-30
GLD,5492,2005-01-03,2026-10-30
SPY,5492,2005-01-03,2026-10-30
TIP,5492,2005-01-03,2026-10-30
TLT,5492,2005-01-03,2026-10-30
VNQ,5492,2005-01-03,2026-10-30


In [86]:
duckdb.sql("""
    SELECT 
        date,
        ticker,
        count(*) as n
    FROM stg_prices_preview
    GROUP BY date, ticker 
    HAVING COUNT(*) > 1 
""")

dateDATE,tickerVARCHAR,nBIGINT


## Sanity checks on the staging preview

Row counts per ticker match the trading calendar, DBC starts on 2006-02-06,
and `(date, ticker)` has no duplicates. These checks became `src/validate.py`.

In [87]:

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"

conn = duckdb.connect(str(DB_PATH), read_only=True)

In [ ]:
# 1. No duplicates values in stg_prices /// severity: error 

conn.execute("""
    SELECT COUNT(*) FROM (
        SELECT date, ticker
        FROM stg_prices
        GROUP BY date, ticker
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

0

In [ ]:
# 2. No nulls values in stg_prices /// severity: error

conn.execute("""
    SELECT COUNT(*) 
        FROM stg_prices
        WHERE date IS NULL 
        OR ticker IS NULL
        OR adj_close IS NULL
""").fetchone()[0]

0

In [ ]:
# 3. Close and adj_close are positive in stg_prices /// severity: error

conn.execute("""
    SELECT COUNT(*)
        FROM stg_prices
        WHERE close <= 0 AND adj_close <= 0
    """).fetchone()[0]

0

In [ ]:
# 4. all 6 tickers are present in stg_prices /// severity: error

conn.execute("""
    SELECT 6 - COUNT(DISTINCT(ticker))
    FROM stg_prices
    """).fetchone()[0]

0

In [ ]:
# 5. Data coherence in stg_prices /// severity: warning

conn.execute("""
    SELECT COUNT(*)
    FROM stg_prices
    WHERE close NOT BETWEEN low and high
""").fetchone()[0]

0

## Largest daily moves

The largest single-day move is −19.5% (VNQ, 2008-12-01). The top 10 are all
genuine crisis days (late 2008, March 2009, March 2020), and VNQ accounts
for 9 of them: real estate was the most volatile asset during a housing-driven crisis.
The extreme-move warning in `validate.py` uses a 25% threshold, so it flags
data errors (e.g. unadjusted splits) rather than real market crashes.

In [121]:
# 6. Extreme movements in stg_prices /// severity: warning

conn.execute("""
    SELECT COUNT(*)
    FROM(
        SELECT 
        ticker,
        date,
        adj_close,
        adj_close / LAG(adj_close) OVER (partition by ticker ORDER BY date) - 1 AS profitability
        FROM stg_prices
        ) AS sub
    WHERE ABS(sub.profitability) > 0.25
""").fetchone()[0]

0

In [ ]:
# 7. Data freshness in stg_prices /// severity: warning

conn.execute("""
    SELECT COUNT(*)
    FROM (
        SELECT ticker
        FROM stg_prices
        GROUP BY ticker
        HAVING date_diff('day', MAX(date), CURRENT_DATE) > 7
        )
""").fetchone()[0]

0

In [ ]:
# 8. Duplicate values in stg_fred /// severity: error

conn.execute("""
    SELECT COUNT(*) FROM (
        SELECT date, series_id
        FROM stg_fred
        GROUP BY date, series_id
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

0

In [ ]:
# 9. Null values in stg_fred /// severity: error

conn.execute("""
    SELECT COUNT(*)
    FROM stg_fred
    WHERE value IS NULL  
    AND NOT (series_id = 'CPIAUCSL' AND date = '2025-10-01')
""").fetchone()[0]

0

In [ ]:
# 10. All FRED's dates are 1 /// severity: error

conn.execute("""
    SELECT COUNT(*)
    FROM stg_fred
    WHERE day(date) != 1
""").fetchone()[0]

0

In [126]:
# 11. No months missing in stg_fred /// severity: error

conn.execute("""
    SELECT COUNT(*)
    FROM(
        SELECT            
            series_id,
            date,
            value,
            LAG(date) OVER (PARTITION BY series_id ORDER BY date) AS prev_month
        FROM stg_fred)
    WHERE date_diff('month',prev_month ,date ) <> 1
""").fetchone()[0]

0